# EDSR ×3 — 미리 학습된 모델 적용

흐린 위성사진(10 m) → 3배 선명하게(3.33 m). 학습 없이 완성된 가중치만 쓴다. GPU 없어도 됩니다.

## 1. 데이터

In [ ]:
import urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'sr_models.py', 'sr_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 모델 적용

In [ ]:
import torch
from sr_models import load_edsr

net = load_edsr(fetch(f'{BASE}/models/03_edsr_x3/checkpoints/edsr_x3.pt', 'edsr_x3.pt'))

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device)
    return net(t).clamp(0, 255).round()[0].cpu().numpy().transpose(1, 2, 0).astype('uint8')

print('EDSR x3 준비 완료')

## 3. 정량 평가

In [ ]:
rows = compare(upscale, label='EDSR')

## 4. 결과

In [ ]:
show_results(upscale, 'EDSR', center=[(180, 300), (300, 80)])
show_test(upscale, 'EDSR', center=(800, 800), size=140)